# E06 S04 shock_prompt_ablation

Controlled prompt-ablation diagnostic for `shock_war`. This notebook creates S04 artifacts only under `data/runs/e06_shock_prompt_ablation/` and does not modify production code, DB files, or existing S02/S03 artifacts.

Executing the generation and sampling cells calls OpenRouter. Validation should only inspect notebook structure.


In [1]:
from pathlib import Path

POINT_LABEL = "shock_war"
RUN_DATE = "2022-03-25"
TRUE_YEAR = 2022
TRUE_MONTH = 3

SOURCE_RUN_DIR = Path("data/runs/e06_cycle_model_bench/gpt54__qwen397b/shock_war")
OUT_DIR = Path("data/runs/e06_shock_prompt_ablation")

MODEL_N = "openai/gpt-5.4"
MODEL_J1 = "qwen/qwen3.5-397b-a17b"
MODEL_J2 = "anthropic/claude-haiku-4.5"

N_TEMPERATURE = 0.3
J1_TEMPERATURE = 0.1
J2_TEMPERATURE = 0.2
J2_N_SAMPLES = 5
J2_JSON_REPAIR_ATTEMPTS = 2

Y_FLOOR = 0.70
JUDGE_BLIND_THRESHOLD = 0.40
SCORING_VERSION = "period_aware_v2"


In [2]:
import json
import os
import re
import time
from collections import Counter
from statistics import mean
from typing import Any, Optional

import yaml
from dotenv import load_dotenv
from openai import OpenAI

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"


def find_project_root(anchor_path: Path) -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / anchor_path).exists():
            return candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / anchor_path).exists():
            return candidate
    raise FileNotFoundError(f"Could not locate project root containing {anchor_path}")


PROJECT_ROOT = find_project_root(SOURCE_RUN_DIR)
load_dotenv(PROJECT_ROOT / ".env")
SOURCE_RUN_DIR = SOURCE_RUN_DIR if SOURCE_RUN_DIR.is_absolute() else PROJECT_ROOT / SOURCE_RUN_DIR
OUT_DIR = OUT_DIR if OUT_DIR.is_absolute() else PROJECT_ROOT / OUT_DIR


def relative_path(path: Path | None) -> str | None:
    if path is None:
        return None
    try:
        return str(Path(path).relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def read_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
        f.write("\n")


def read_text(path: Path) -> str:
    if not path.exists():
        raise FileNotFoundError(f"Missing text artifact: {path}")
    return path.read_text(encoding="utf-8")


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def resolve_final_candidate_path(path_value: str | Path) -> Path:
    path = Path(path_value)
    candidates = [path] if path.is_absolute() else [PROJECT_ROOT / path, SOURCE_RUN_DIR / path]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("C1 accepted candidate does not exist. Searched: " + ", ".join(map(str, candidates)))


def call_llm(model: str, system_prompt: str, user_prompt: str, temperature: float) -> str:
    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is not set in the environment")

    last_error: Exception | None = None
    for attempt in range(1, 4):
        try:
            client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=api_key)
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=temperature,
            )
            content = response.choices[0].message.content
            if content and content.strip():
                return content.strip()
            last_error = RuntimeError(f"Model {model} returned empty content")
        except Exception as exc:
            last_error = exc
        if attempt < 3:
            time.sleep(2 * attempt)
    raise RuntimeError(f"OpenRouter call failed after 3 attempts for model={model}: {last_error}") from last_error


def extract_json(raw_text: str) -> dict[str, Any]:
    text = raw_text.strip()
    fence_match = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    if fence_match:
        text = fence_match.group(1).strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError as exc:
        try:
            parsed = yaml.safe_load(text)
        except yaml.YAMLError as yaml_exc:
            raise ValueError(f"Could not parse JSON/YAML after optional fence stripping: json={exc}; yaml={yaml_exc}\nPreview:\n{text[:500]}") from exc
    if not isinstance(parsed, dict):
        raise ValueError(f"Expected a JSON object, got {type(parsed).__name__}")
    return parsed


In [3]:
Q3_SIGNALS_SYSTEM = """
You are J1, an in-loop forecast-signal judge for a Russian macroeconomic summary.

Task: extract only forecast-relevant inflation expectation signals that are recoverable from the summary.
Return strict JSON only. Do not include markdown, commentary, or extra keys.

Output schema:
{
  "signals": [
    {
      "id": "sig_001",
      "category": "monetary_policy | supply_shock_food | supply_shock_energy | currency_pressure | fiscal_expansion | demand_shift | regulated_prices | inflation_print",
      "direction": "up | down | mixed | neutral",
      "strength": 1,
      "anchor_phrase": "..."
    }
  ],
  "summary_direction": "up | down | mixed | neutral",
  "summary_confidence": 0.0
}

Use the closed category list only:
- monetary_policy
- supply_shock_food
- supply_shock_energy
- currency_pressure
- fiscal_expansion
- demand_shift
- regulated_prices
- inflation_print

No extra categories are allowed.
direction must be exactly one of: up, down, mixed, neutral.

Direction conventions:
- monetary_policy up means tightening pressure: key-rate increase, expected key-rate increase, higher policy-rate path, or tighter lending / macroprudential stance if inflationary pressure is the reason.
- monetary_policy down means easing pressure: key-rate decrease, expected key-rate decrease, softer policy path, or credit easing.
- monetary_policy neutral means no directional monetary-policy signal.
- monetary_policy mixed means both tightening and easing signals are present.
- regulated_prices down means administrative price containment, tariffs frozen, import-duty relief, or explicit anti-price measures.
- regulated_prices up means regulated/tariff prices increase.
- regulated_prices mixed means both containment and increases appear.

strength must be an integer from 1 to 3, where 1=weak, 2=medium, 3=strong.
Extract only signals actually present in the text; do not infer hidden signals from general macro knowledge.
Do not reward the rewriter for adding new signals.
anchor_phrase must be short and traceable to the text, copied or minimally normalized from the summary.
summary_confidence must be a number from 0.0 to 1.0.
Use stable ids sig_001, sig_002, ... .
""".strip()

BASE_N_REWRITER_SYSTEM = """
You are N, a Russian-language neutralizing rewriter for a blind macroeconomic forecasting experiment.

Rewrite the provided summary so that it preserves forecast-relevant inflation expectation signals while reducing historical identifiability.
Return only the rewritten summary as plain text. Do not return JSON. Do not add commentary.

Preserve economic signal direction and approximate strength. Do not invent new facts, new chronology, or new causal claims.
Remove or abstract period-identifying details when they are not needed for signal preservation.
""".strip()

MECHANISM_ABSTRACTION_APPENDIX = """
For event-driven shocks, do not preserve actor-event structure.

Abstract named geopolitical, sanction, military, and cross-border financial events into mechanism-class descriptions.

Preserve:
- direction of currency pressure;
- severity of external financial/trade shock;
- inflationary transmission channel;
- direction of monetary/regulatory response;
- pressure on household expectations.

Do not preserve:
- actor blocs;
- country alignment;
- reserve-freeze structure;
- sanction/import-ban vocabulary;
- unique policy sequence;
- named conflict references.
""".strip()

AGGRESSIVE_MECHANISM_ABSTRACTION_APPENDIX = """
The output must not contain words or roots derived from:
санкц, военн, конфликт, Украина, Россия, США, ЕС, Запад,
резерв, SWIFT, импортный запрет, экспортное ограничение,
недружественные страны, газ за рубли.

If these concepts are needed for signal preservation, replace them with generic mechanism-class phrases:
- "резкий внешний финансово-торговый шок";
- "внешние ограничения";
- "разрыв части внешнеэкономических связей";
- "усиление валютной и инфляционной неопределённости";
- "экстренная стабилизационная реакция финансовых властей".

Do not preserve a unique sequence of measures if the sequence identifies the historical episode.
Preserve only the economic transmission mechanism and signal direction/strength.
""".strip()

N_C4_COORDINATE_FREE_MECHANISM_SYSTEM = """
For this condition, rewrite the shock at coordinate-free mechanism level.

Do not preserve:
- actor-event structure;
- country/bloc alignment;
- sanction/import-ban vocabulary;
- reserve-freeze structure;
- named conflict references;
- unique policy sequence;
- exact or near-exact numeric coordinates;
- historically distinctive policy levels;
- historically distinctive price/exchange-rate levels;
- unique consumer panic product anchors.

Preserve only:
- direction of currency pressure;
- severity class of external financial/trade shock;
- inflationary transmission channel;
- direction of monetary/regulatory response;
- pressure on household inflation expectations;
- broad sectoral channel when needed.

Replace exact or distinctive numeric coordinates with scale-preserving qualitative language:
- "двузначный уровень" instead of exact policy-rate/inflation levels;
- "резкое ослабление национальной валюты" instead of exact FX levels;
- "сильное ухудшение прогноза" instead of exact forecast values;
- "экстренные стабилизационные меры" instead of exact sequence of policy measures;
- "отдельные товары повседневного спроса" instead of product-specific panic anchors such as sugar.

The output must not contain:
- exact percentages such as 20%, 8%, 12%, etc.;
- exact FX levels or "above/below 100 rubles";
- exact key-rate level;
- exact GDP forecast levels;
- exact liquidity/fiscal amounts;
- commodity benchmark names: Brent, WTI, Urals;
- product-specific panic anchors: sugar / сахар if used as a historical fingerprint;
- "санкции", "санкционный", "военный", "конфликт", "Украина", "Россия", "США", "ЕС", "Запад", "резервы", "SWIFT", "недружественные страны", "газ за рубли".

Do not make the text generic or content-free. Preserve forecast-relevant signal at mechanism level.
If removing a numeric coordinate would weaken the signal, replace it with a severity class rather than deleting the fact.
If removing an event label would weaken the signal, replace it with a mechanism-class description rather than deleting the causal channel.
""".strip()

N_PROMPT_C2 = "\n\n".join([BASE_N_REWRITER_SYSTEM, MECHANISM_ABSTRACTION_APPENDIX])
N_PROMPT_C3 = "\n\n".join([BASE_N_REWRITER_SYSTEM, MECHANISM_ABSTRACTION_APPENDIX, AGGRESSIVE_MECHANISM_ABSTRACTION_APPENDIX])
N_PROMPT_C4 = "\n\n".join([BASE_N_REWRITER_SYSTEM, MECHANISM_ABSTRACTION_APPENDIX, N_C4_COORDINATE_FREE_MECHANISM_SYSTEM])

J2_IDENTIFIABILITY_SYSTEM = """
You are an independent holdout judge for a blind identifiability evaluation.

Your task is to infer the historical period described by the provided text alone.
The text may be raw or deliberately neutralized to remove identifying markers.

Rules:
- Rely only on the content of the text.
- Do not rely on metadata, filenames, labels, user context, evaluation setup, or any hidden assumptions about why the text was selected.
- Do not assume there is a fixed list of allowed periods.
- If the text is ambiguous, report calibrated uncertainty instead of forcing an answer.
- Return strict JSON only. Do not wrap it in prose.

Return exactly this JSON shape:
{
  "can_identify_period": true,
  "predicted_year": 2022,
  "predicted_month": 3,
  "predicted_period_description": "...",
  "confidence": 0.0,
  "evidence_type": "explicit_marker | macro_configuration | symbolic_statement | absence_pattern | guess | cannot_determine",
  "rationale": "...",
  "identifying_evidence": ["..."]
}

Field rules:
- can_identify_period must be a boolean.
- predicted_year must be an integer or null.
- predicted_month must be an integer from 1 to 12 or null.
- confidence must be a float from 0 to 1.
- evidence_type must be exactly one of: explicit_marker, macro_configuration, symbolic_statement, absence_pattern, guess, cannot_determine.
- identifying_evidence must be a list of short evidence phrases from the text.
- If you cannot identify the period, set can_identify_period to false, predicted_year to null, predicted_month to null, confidence to 0.4 or lower, and evidence_type to cannot_determine or guess.
""".strip()


def make_n_user_prompt(raw_summary: str) -> str:
    return "Rewrite the following summary under the system instructions.\n\nSummary:\n" + raw_summary


def make_j2_user_prompt(summary_text: str) -> str:
    return "Infer the historical period from the following text alone. Return strict JSON using the required schema.\n\nTEXT:\n" + summary_text


def make_j2_json_repair_prompt(raw_response: str, error: Exception) -> str:
    return f"""
Your previous response could not be parsed as valid JSON for this schema.
Return the same judgment again as strict valid JSON only, with no markdown fence, no commentary, and no extra keys.

Required JSON shape:
{
  "can_identify_period": true,
  "predicted_year": 2022,
  "predicted_month": 3,
  "predicted_period_description": "...",
  "confidence": 0.0,
  "evidence_type": "explicit_marker | macro_configuration | symbolic_statement | absence_pattern | guess | cannot_determine",
  "rationale": "...",
  "identifying_evidence": ["..."]
}

Parsing/validation error:
{error}

Previous response:
{raw_response}
""".strip()


In [4]:
Q3_CATEGORIES = {
    "monetary_policy", "supply_shock_food", "supply_shock_energy", "currency_pressure",
    "fiscal_expansion", "demand_shift", "regulated_prices", "inflation_print",
}
Q3_DIRECTIONS = {"up", "down", "mixed", "neutral"}
EVIDENCE_TYPES = {"explicit_marker", "macro_configuration", "symbolic_statement", "absence_pattern", "guess", "cannot_determine"}
REQUIRED_J2_FIELDS = {"can_identify_period", "predicted_year", "predicted_month", "predicted_period_description", "confidence", "evidence_type", "rationale", "identifying_evidence"}
PERIOD_WEIGHTS = {"exact_month": 1.0, "adjacent_month": 0.75, "same_quarter": 0.5, "same_year": 0.25, "wrong_period": 0.0, "cannot_determine": 0.0}

MONTH_PATTERN = r"\b(?:январ[ьяе]|феврал[ьяе]|март[ае]?|апрел[ьяе]|ма[йяюе]|июн[ьяе]|июл[ьяе]|август[ае]?|сентябр[ьяе]|октябр[ьяе]|ноябр[ьяе]|декабр[ьяе])\b"
RESIDUAL_HARD_PATTERNS = [
    r"\b20\d{2}\b|\b19\d{2}\b", MONTH_PATTERN, r"COVID|ковид|коронавирус|пандем",
    r"\bинФОМ\b|Банк\s+России|\bЦБ\b|Центробанк", r"Citi|Пят[её]роч|ФРГ|Brent|WTI|Urals|Донбасс|спецоперац|контрсанкц",
    r"санкц", r"Украин", r"Россия", r"США", r"\bЕС\b", r"Запад", r"резерв", r"SWIFT", r"недружествен",
    r"импортн\w*\s+запрет", r"экспортн\w*\s+огранич",
    r"газ\w*.{0,80}рубл", r"оплат\w*.{0,80}газ\w*.{0,80}рубл", r"газ\w*\s+за\s+рубл",
    r"\b20\s*%", r"\b\d{2,3}\s*%", r"\b1\d{2}\s*руб",
    r"ключев\w+\s+ставк\w*.{0,80}\b20\s*%", r"ставк\w*.{0,80}\b20\s*%", r"инфляци\w*.{0,80}\b20\s*%",
    r"ВВП.{0,80}-\s*\d", r"\b\d+[,.]?\d*\s*трлн", r"\b\d+[,.]?\d*\s*млрд", r"Brent|WTI|Urals", r"сахар",
]
RESIDUAL_WARN_PATTERNS = [
    r"историческ\w+ максимум", r"годов\w+ минимум", r"стагфляц", r"голландск\w+ болезн", r"сырьев\w+ суперцикл",
    r"за три квартала", r"овсян\w+ хлоп", r"бюджетн\w+ правил", r"резервн\w+ валют", r"газ\w* в Европе.*ключев\w+ триггер",
    r"посткризисн\w+ волатильност", r"глава центробанка.*сбережен", r"хранит сбережен", r"заморозк\w+ резерв", r"параллельн\w+ импорт",
    r"валютн\w+ шок", r"внешн\w+ огранич", r"геополит", r"энергетическ\w+ кризис", r"капитал\w+ контрол", r"платежн\w+ систем",
]


def validate_q3(parsed: dict[str, Any]) -> None:
    signals = parsed.get("signals")
    if not isinstance(signals, list):
        raise ValueError("Q3 JSON must contain a signals list")
    for signal in signals:
        if signal.get("category") not in Q3_CATEGORIES:
            raise ValueError(f"Invalid Q3 category: {signal.get('category')}")
        if signal.get("direction") not in Q3_DIRECTIONS:
            raise ValueError(f"Invalid Q3 direction: {signal.get('direction')}")
        if not isinstance(signal.get("strength"), int) or not 1 <= signal["strength"] <= 3:
            raise ValueError(f"Invalid Q3 strength: {signal.get('strength')}")
        if not isinstance(signal.get("anchor_phrase"), str) or not signal["anchor_phrase"].strip():
            raise ValueError("Q3 anchor_phrase must be a non-empty string")


def q3_preservation_score(baseline: dict, candidate: dict) -> dict:
    baseline_signals = baseline.get("signals", [])
    candidate_signals = candidate.get("signals", [])
    candidate_by_category: dict[str, list[dict[str, Any]]] = {}
    for signal in candidate_signals:
        candidate_by_category.setdefault(signal.get("category"), []).append(signal)

    matched_categories, missing_categories = [], []
    direction_matches = strength_consistent = strength_upgrades = strength_downgrades = 0
    for base in baseline_signals:
        category = base.get("category")
        options = candidate_by_category.get(category, [])
        if not options:
            missing_categories.append(category)
            continue
        matched_categories.append(category)
        exact_direction = [opt for opt in options if opt.get("direction") == base.get("direction")]
        if not exact_direction:
            continue
        direction_matches += 1
        chosen = min(exact_direction, key=lambda opt: abs(int(opt.get("strength", 0)) - int(base.get("strength", 0))))
        diff = int(chosen.get("strength", 0)) - int(base.get("strength", 0))
        if abs(diff) <= 1:
            strength_consistent += 1
        if diff > 0:
            strength_upgrades += 1
        elif diff < 0:
            strength_downgrades += 1

    baseline_count = len(baseline_signals)
    matched_count = len(matched_categories)
    recall = matched_count / baseline_count if baseline_count else 0.0
    direction_rate = direction_matches / matched_count if matched_count else 0.0
    strength_rate = strength_consistent / direction_matches if direction_matches else 0.0
    baseline_categories = {s.get("category") for s in baseline_signals}
    candidate_categories = {s.get("category") for s in candidate_signals}
    return {
        "q3_preservation": round(recall * direction_rate * strength_rate, 6),
        "q3_strength_upgrades": strength_upgrades,
        "q3_strength_downgrades": strength_downgrades,
        "q3_strength_net_shift": strength_upgrades - strength_downgrades,
        "matched_categories": sorted(set(matched_categories)),
        "missing_categories": sorted(set(missing_categories)),
        "extra_categories": sorted(candidate_categories - baseline_categories),
    }


def residual_fingerprint_check(text: str) -> dict[str, Any]:
    def collect(patterns: list[str]) -> list[dict[str, Any]]:
        out = []
        for pattern in patterns:
            for match in re.finditer(pattern, text, flags=re.IGNORECASE | re.DOTALL):
                out.append({"pattern": pattern, "match": match.group(0), "span": [match.start(), match.end()]})
        return out
    hard_fail = collect(RESIDUAL_HARD_PATTERNS)
    warn = collect(RESIDUAL_WARN_PATTERNS)
    return {"hard_fail": hard_fail, "warn": warn, "manual_fail": bool(hard_fail)}


def validate_j2_parsed(parsed: dict[str, Any]) -> None:
    missing = sorted(REQUIRED_J2_FIELDS - set(parsed))
    if missing:
        raise ValueError(f"J2 JSON response is missing fields: {missing}")
    if not isinstance(parsed["can_identify_period"], bool):
        raise TypeError("can_identify_period must be a boolean")
    if parsed["predicted_year"] is not None and not isinstance(parsed["predicted_year"], int):
        raise TypeError("predicted_year must be an integer or null")
    if parsed["predicted_month"] is not None and (not isinstance(parsed["predicted_month"], int) or not 1 <= parsed["predicted_month"] <= 12):
        raise ValueError("predicted_month must be an integer from 1 to 12 or null")
    if not isinstance(parsed["confidence"], (int, float)):
        raise TypeError("confidence must be numeric")
    if parsed["evidence_type"] not in EVIDENCE_TYPES:
        raise ValueError(f"Invalid evidence_type: {parsed['evidence_type']}")
    if not isinstance(parsed["identifying_evidence"], list):
        raise TypeError("identifying_evidence must be a list")


def clamp_confidence(value: Any) -> float:
    return max(0.0, min(1.0, float(value)))


def month_index(year: int, month: int) -> int:
    return year * 12 + month


def month_distance(pred_year: Optional[int], pred_month: Optional[int], true_year: int, true_month: int) -> Optional[int]:
    if pred_year is None or pred_month is None:
        return None
    if not (1 <= int(pred_month) <= 12):
        return None
    return abs(month_index(int(pred_year), int(pred_month)) - month_index(int(true_year), int(true_month)))


def same_calendar_quarter(year_a: int, month_a: int, year_b: int, month_b: int) -> bool:
    return year_a == year_b and (month_a - 1) // 3 == (month_b - 1) // 3


def score_identification(parsed: dict[str, Any], true_year: int, true_month: int) -> dict[str, Any]:
    confidence = clamp_confidence(parsed.get("confidence", 0.0))
    can_identify = parsed.get("can_identify_period") is True
    pred_year, pred_month = parsed.get("predicted_year"), parsed.get("predicted_month")
    dist = month_distance(pred_year, pred_month, true_year, true_month) if can_identify else None
    exact_success = bool(can_identify and pred_year == true_year and pred_month == true_month)
    adjacent_month_success = bool(can_identify and dist is not None and dist <= 1)
    same_quarter_success = bool(can_identify and pred_year is not None and pred_month is not None and same_calendar_quarter(int(pred_year), int(pred_month), true_year, true_month))
    same_year_success = bool(can_identify and pred_year == true_year)
    if not can_identify or pred_year is None or pred_month is None:
        period_success_level = "cannot_determine"
    elif exact_success:
        period_success_level = "exact_month"
    elif adjacent_month_success:
        period_success_level = "adjacent_month"
    elif same_quarter_success:
        period_success_level = "same_quarter"
    elif same_year_success:
        period_success_level = "same_year"
    else:
        period_success_level = "wrong_period"
    period_weight = PERIOD_WEIGHTS[period_success_level]
    return {
        "exact_success": exact_success,
        "adjacent_month_success": adjacent_month_success,
        "same_quarter_success": same_quarter_success,
        "same_year_success": same_year_success,
        "period_success_level": period_success_level,
        "period_weight": period_weight,
        "weighted_success": confidence if exact_success else 0.0,
        "period_weighted_success": round(confidence * period_weight, 6),
        "confidence": confidence,
        "true_year": true_year,
        "true_month": true_month,
        "predicted_year": pred_year,
        "predicted_month": pred_month,
        "month_distance": dist,
        "scoring_version": SCORING_VERSION,
    }


In [5]:
def run_q3(summary_text: str) -> dict[str, Any]:
    raw_response = call_llm(MODEL_J1, Q3_SIGNALS_SYSTEM, f"Summary:\n\n{summary_text}", J1_TEMPERATURE)
    parsed = extract_json(raw_response)
    validate_q3(parsed)
    return parsed


def load_inputs() -> dict[str, Any]:
    raw_path = SOURCE_RUN_DIR / "iter_00_raw.md"
    final_record_path = SOURCE_RUN_DIR / "final_record.json"
    if not raw_path.exists():
        raise FileNotFoundError(f"Missing C0 raw summary: {raw_path}")
    if not final_record_path.exists():
        raise FileNotFoundError(f"Missing C1 final_record.json: {final_record_path}")
    final_record = read_json(final_record_path)
    if "final_candidate_path" not in final_record:
        raise KeyError("final_record.json is missing final_candidate_path")
    final_candidate_path = resolve_final_candidate_path(final_record["final_candidate_path"])
    return {
        "raw_path": raw_path,
        "raw_summary": read_text(raw_path),
        "final_record_path": final_record_path,
        "final_record": final_record,
        "final_candidate_path": final_candidate_path,
        "final_candidate_text": read_text(final_candidate_path),
    }


def prepare_conditions() -> tuple[list[dict[str, Any]], dict[str, Any]]:
    inputs = load_inputs()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    q3_baseline = run_q3(inputs["raw_summary"])
    q3_baseline_path = OUT_DIR / "q3_baseline_raw.json"
    write_json(q3_baseline_path, q3_baseline)

    specs = [
        ("C0_raw", "input.md", inputs["raw_summary"], inputs["raw_path"], "raw", None, False),
        ("C1_current_best", "input.md", inputs["final_candidate_text"], inputs["final_candidate_path"], "accepted_final_candidate", None, True),
        ("C2_mechanism_abstraction", "candidate.md", call_llm(MODEL_N, N_PROMPT_C2, make_n_user_prompt(inputs["raw_summary"]), N_TEMPERATURE), inputs["raw_path"], "single_n_rewrite_from_raw", "mechanism_abstraction", True),
        ("C3_aggressive_mechanism_abstraction", "candidate.md", call_llm(MODEL_N, N_PROMPT_C3, make_n_user_prompt(inputs["raw_summary"]), N_TEMPERATURE), inputs["raw_path"], "single_n_rewrite_from_raw", "aggressive_mechanism_abstraction", True),
        ("C4_coordinate_free_mechanism_abstraction", "candidate.md", call_llm(MODEL_N, N_PROMPT_C4, make_n_user_prompt(inputs["raw_summary"]), N_TEMPERATURE), inputs["raw_path"], "single_n_rewrite_from_raw", "coordinate_free_mechanism_abstraction", True),
    ]

    records = []
    for condition, filename, text, source_path, source_kind, prompt_variant, needs_checks in specs:
        condition_dir = OUT_DIR / condition
        text_path = condition_dir / filename
        write_text(text_path, text)
        q3_path = residual_path = None
        q3_score = None
        residual = None
        if needs_checks:
            q3_candidate = run_q3(text)
            q3_path = condition_dir / "q3_candidate.json"
            write_json(q3_path, q3_candidate)
            q3_score = q3_preservation_score(q3_baseline, q3_candidate)
            residual = residual_fingerprint_check(text)
            residual_path = condition_dir / "residual_check.json"
            write_json(residual_path, residual)

        record = {
            "condition": condition,
            "point_label": POINT_LABEL,
            "run_date": RUN_DATE,
            "true_year": TRUE_YEAR,
            "true_month": TRUE_MONTH,
            "source_kind": source_kind,
            "source_path": relative_path(source_path),
            "text_path": relative_path(text_path),
            "q3_baseline_path": relative_path(q3_baseline_path),
            "q3_candidate_path": relative_path(q3_path),
            "residual_check_path": relative_path(residual_path),
            "q3_preservation": q3_score["q3_preservation"] if q3_score else None,
            "q3_strength_upgrades": q3_score["q3_strength_upgrades"] if q3_score else None,
            "q3_strength_downgrades": q3_score["q3_strength_downgrades"] if q3_score else None,
            "q3_strength_net_shift": q3_score["q3_strength_net_shift"] if q3_score else None,
            "matched_categories": q3_score["matched_categories"] if q3_score else None,
            "missing_categories": q3_score["missing_categories"] if q3_score else None,
            "extra_categories": q3_score["extra_categories"] if q3_score else None,
            "residual_hard_fail_count": len(residual["hard_fail"]) if residual else None,
            "residual_warn_count": len(residual["warn"]) if residual else None,
            "residual_manual_fail": residual["manual_fail"] if residual else None,
            "model_roles": {"N": MODEL_N, "J1": MODEL_J1, "J2": MODEL_J2},
            "n_prompt_variant": prompt_variant,
        }
        write_json(condition_dir / "condition_record.json", record)
        records.append({**record, "_text": text, "_condition_dir": condition_dir})

    return records, {
        "inputs": {
            "raw_path": relative_path(inputs["raw_path"]),
            "final_record_path": relative_path(inputs["final_record_path"]),
            "final_candidate_path": relative_path(inputs["final_candidate_path"]),
        },
        "q3_baseline_path": relative_path(q3_baseline_path),
        "q3_baseline_raw": q3_baseline,
    }


In [6]:
def aggregate_j2_samples(condition: str, samples: list[dict[str, Any]]) -> dict[str, Any]:
    if not samples:
        raise ValueError("Cannot aggregate zero J2 samples")
    scoring = [sample["scoring"] for sample in samples]
    return {
        "condition": condition,
        "n_samples": len(samples),
        "exact_id_score": mean(row["weighted_success"] for row in scoring),
        "period_id_score": mean(row["period_weighted_success"] for row in scoring),
        "period_delta_vs_raw": 0.0,
        "mean_confidence": mean(row["confidence"] for row in scoring),
        "evidence_type_counts": dict(sorted(Counter(sample["parsed"].get("evidence_type") for sample in samples).items())),
        "period_success_level_counts": dict(sorted(Counter(row["period_success_level"] for row in scoring).items())),
        "scoring_version": SCORING_VERSION,
    }


def run_j2_samples(record: dict[str, Any]) -> dict[str, Any]:
    sample_dir = record["_condition_dir"] / "j2_samples"
    samples = []
    for sample_index in range(1, J2_N_SAMPLES + 1):
        raw_response = call_llm(MODEL_J2, J2_IDENTIFIABILITY_SYSTEM, make_j2_user_prompt(record["_text"]), J2_TEMPERATURE)
        last_error = None
        for repair_attempt in range(J2_JSON_REPAIR_ATTEMPTS + 1):
            if repair_attempt:
                raw_response = call_llm(MODEL_J2, J2_IDENTIFIABILITY_SYSTEM, make_j2_json_repair_prompt(raw_response, last_error), J2_TEMPERATURE)
            try:
                parsed = extract_json(raw_response)
                validate_j2_parsed(parsed)
                break
            except (TypeError, ValueError) as exc:
                last_error = exc
        else:
            raise ValueError(f"J2 returned invalid JSON after {J2_JSON_REPAIR_ATTEMPTS} repair attempts for condition={record['condition']} sample={sample_index}") from last_error
        sample = {
            "condition": record["condition"],
            "sample_index": sample_index,
            "model": MODEL_J2,
            "temperature": J2_TEMPERATURE,
            "input_text_path": record["text_path"],
            "raw_response": raw_response,
            "parsed": parsed,
            "scoring": score_identification(parsed, TRUE_YEAR, TRUE_MONTH),
        }
        write_json(sample_dir / f"sample_{sample_index:02d}.json", sample)
        samples.append(sample)
    aggregate = aggregate_j2_samples(record["condition"], samples)
    write_json(record["_condition_dir"] / "j2_aggregate.json", aggregate)
    return aggregate


def interpretation_case(row: dict[str, Any]) -> str:
    if row["condition"] == "C0_raw":
        return "baseline_raw"
    q3 = row.get("q3_preservation")
    delta = row.get("period_delta_vs_raw")
    hard = row.get("residual_hard_fail_count")
    if q3 is None or delta is None or hard is None:
        return "mixed_or_inconclusive"
    if q3 >= Y_FLOOR and delta >= 0.40 and hard == 0:
        return "Case A - prompt failure confirmed"
    if q3 < Y_FLOOR and delta >= 0.40:
        return "Case B - ceiling / signal trade-off"
    if q3 >= Y_FLOOR and delta < 0.20:
        return "Case C - method ceiling"
    if q3 < Y_FLOOR and delta < 0.20:
        return "Case D - bad prompt"
    return "mixed_or_inconclusive"


def build_comparison_rows(records: list[dict[str, Any]], aggregates: dict[str, dict[str, Any]]) -> list[dict[str, Any]]:
    raw_period_score = aggregates["C0_raw"]["period_id_score"]
    rows = []
    for record in records:
        condition = record["condition"]
        aggregate = aggregates[condition]
        aggregate["period_delta_vs_raw"] = raw_period_score - aggregate["period_id_score"]
        write_json(record["_condition_dir"] / "j2_aggregate.json", aggregate)
        row = {
            "condition": condition,
            "q3_preservation": record.get("q3_preservation"),
            "q3_strength_net_shift": record.get("q3_strength_net_shift"),
            "residual_hard_fail_count": record.get("residual_hard_fail_count"),
            "residual_warn_count": record.get("residual_warn_count"),
            "exact_id_score": aggregate["exact_id_score"],
            "period_id_score": aggregate["period_id_score"],
            "period_delta_vs_raw": aggregate["period_delta_vs_raw"],
            "mean_confidence": aggregate["mean_confidence"],
            "evidence_type_counts": aggregate["evidence_type_counts"],
            "period_success_level_counts": aggregate["period_success_level_counts"],
        }
        row["interpretation_case"] = interpretation_case(row)
        rows.append(row)
    return rows


def choose_final_interpretation(rows: list[dict[str, Any]]) -> dict[str, Any]:
    case_counts = Counter(row["interpretation_case"] for row in rows if row["condition"] != "C0_raw")
    priority = ["Case A - prompt failure confirmed", "Case C - method ceiling", "Case B - ceiling / signal trade-off", "Case D - bad prompt", "mixed_or_inconclusive"]
    selected = next((case for case in priority if case_counts.get(case)), "mixed_or_inconclusive")
    return {"selected_interpretation": selected, "case_counts": dict(case_counts)}


condition_records, setup_record = prepare_conditions()
j2_aggregates = {record["condition"]: run_j2_samples(record) for record in condition_records}
comparison_rows = build_comparison_rows(condition_records, j2_aggregates)
final_interpretation = choose_final_interpretation(comparison_rows)

public_condition_records = []
for record in condition_records:
    public_record = {key: value for key, value in record.items() if not key.startswith("_")}
    public_condition_records.append(public_record)
    write_json(record["_condition_dir"] / "condition_record.json", public_record)

s04_summary = {
    "experiment": "shock_prompt_ablation",
    "scoring_version": SCORING_VERSION,
    "point": {"point_label": POINT_LABEL, "run_date": RUN_DATE, "true_year": TRUE_YEAR, "true_month": TRUE_MONTH},
    "model_config": {"model_n": MODEL_N, "model_j1": MODEL_J1, "model_j2": MODEL_J2, "openrouter_base_url": OPENROUTER_BASE_URL},
    "sampling_config": {"n_temperature": N_TEMPERATURE, "j1_temperature": J1_TEMPERATURE, "j2_temperature": J2_TEMPERATURE, "j2_n_samples": J2_N_SAMPLES, "y_floor": Y_FLOOR, "judge_blind_threshold": JUDGE_BLIND_THRESHOLD},
    "setup_record": setup_record,
    "condition_records": public_condition_records,
    "j2_aggregates": j2_aggregates,
    "final_comparison_rows": comparison_rows,
    "final_interpretation": final_interpretation,
}
write_json(OUT_DIR / "s04_summary.json", s04_summary)


In [7]:
comparison_columns = [
    "condition",
    "q3_preservation",
    "q3_strength_net_shift",
    "residual_hard_fail_count",
    "residual_warn_count",
    "exact_id_score",
    "period_id_score",
    "period_delta_vs_raw",
    "mean_confidence",
    "evidence_type_counts",
    "interpretation_case",
]

if "comparison_rows" not in globals():
    s04_summary = read_json(OUT_DIR / "s04_summary.json")
    comparison_rows = s04_summary["final_comparison_rows"]

try:
    import pandas as pd
    display(pd.DataFrame(comparison_rows)[comparison_columns])
except ImportError:
    print(json.dumps([{key: row.get(key) for key in comparison_columns} for row in comparison_rows], ensure_ascii=False, indent=2))


,condition,q3_preservation,q3_strength_net_shift,residual_hard_fail_count,residual_warn_count,exact_id_score,period_id_score,period_delta_vs_raw,mean_confidence,evidence_type_counts,interpretation_case
0,C0_raw,NaN,NaN,NaN,NaN,0.968,0.968,0.000,0.968,{'explicit_marker': 5},baseline_raw
1,C1_current_best,0.833333,1.0,8.0,2.0,0.920,0.920,0.048,0.920,"{'explicit_marker': 1, 'macro_configuration': 4}",Case C - method ceiling
2,C2_mechanism_abstraction,0.833333,1.0,10.0,1.0,0.920,0.920,0.048,0.920,{'macro_configuration': 5},Case C - method ceiling
3,C3_aggressive_mechanism_abstraction,0.833333,-2.0,14.0,0.0,0.938,0.938,0.030,0.938,{'explicit_marker': 5},Case C - method ceiling
4,C4_coordinate_free_mechanism_abstraction,0.833333,0.0,0.0,0.0,0.920,0.920,0.048,0.920,{'macro_configuration': 5},Case C - method ceiling
